#05 — Bronze XML Ingestion | PySpark Native
Ingests `streets_chunk_4.xml` into `vstone_catalog.bronze.streets_xml`

## Widgets & Configuration

In [0]:
# Added a null guard on the (street_id, date) grain before writing —
#     the first draft wrote whatever came out of the XML reader with no check.
#   - Idempotency is now DELETE (by source file) + append, matching the
#     reference's pattern, instead of a blind mode="overwrite" of the whole
#     table — this matters once streets_xml ever needs to hold more than one
#     source file.
#   - Added the same verification/audit block used in 04_bronze_auto_loader,
#     so all Day 3 bronze notebooks report evidence the same way.

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

# ── Widgets & Configuration ────────────────────────────────────────────────

dbutils.widgets.text("catalog_name", "vstone_catalog", "1. Catalog Name")
dbutils.widgets.text("raw_schema", "raw", "2. Raw Schema")
dbutils.widgets.text("bronze_schema", "bronze", "3. Bronze Schema")
dbutils.widgets.text("chunks_volume", "chunks", "4. Chunks Volume")

CATALOG = dbutils.widgets.get("catalog_name")
RAW_SCHEMA = dbutils.widgets.get("raw_schema")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")
CHUNKS_VOL = dbutils.widgets.get("chunks_volume")

FILE_NAME = "streets_chunk_4.xml"
SOURCE_FILE = f"/Volumes/{CATALOG}/{RAW_SCHEMA}/{CHUNKS_VOL}/{FILE_NAME}"
TARGET_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.streets_xml"

print(f"Source file  : {SOURCE_FILE}")
print(f"Target table : {TARGET_TABLE}") 

## Bronze Schema (all STRING — inferSchema = false)

In [0]:
# rowTag="record" matches what 02_data_chunking wrote this chunk with.
BRONZE_SCHEMA = StructType([
    StructField("noise", StringType(), True),
    StructField("pollution", StringType(), True),
    StructField("date", StringType(), True),
    StructField("light", StringType(), True),
    StructField("raining", StringType(), True),
    StructField("street_id", StringType(), True),
])

print(f"Schema defined — {len(BRONZE_SCHEMA.fields)} columns, inferSchema = false (all STRING)")

## Read XML with PySpark native reader

In [0]:

try:
    df_xml = (spark.read
        .format("xml")
        .option("rowTag", "record")
        .option("inferSchema", "false")
        .schema(BRONZE_SCHEMA)
        .load(SOURCE_FILE)
        .withColumn("load_dt", F.current_timestamp())
        .withColumn("source", F.lit(FILE_NAME)))
except Exception as e:
    raise Exception(
        f"Native XML read failed ({e}). If this workspace lacks native XML "
        f"support (DBR < 14.3), this notebook needs the xml.etree fallback "
        f"reader instead — see 02_data_chunking's own fallback for the "
        f"<record><field>value</field></record> shape it would produce."
    )

raw_count = df_xml.count()
print(f"Rows read from XML: {raw_count:,}")

## Null Guard

In [0]:
# Grain is (street_id, date) — no single "id" column exists here, unlike the
# reference project's car-listing data.

df_clean = df_xml.filter(F.col("street_id").isNotNull() & F.col("date").isNotNull())
null_dropped = raw_count - df_clean.count()
print(f"Rows after null guard: {df_clean.count():,}  (dropped: {null_dropped:,})")

## Idempotent Write: DELETE by source file, then append

In [0]:
if spark.catalog.tableExists(TARGET_TABLE):
    before = spark.table(TARGET_TABLE).count()
    spark.sql(f"DELETE FROM {TARGET_TABLE} WHERE source = '{FILE_NAME}'")
    after_delete = spark.table(TARGET_TABLE).count()
    print(f"Existing rows before delete: {before:,}")
    print(f"Rows deleted for {FILE_NAME}: {before - after_delete:,}")
else:
    print("Table does not exist yet — will be created on write.")

(df_clean.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(TARGET_TABLE))
print("Write complete.")

## Verification & Audit

In [0]:
if not spark.catalog.tableExists(TARGET_TABLE):
    raise Exception(f"Table not found after write: {TARGET_TABLE}")

df_bronze = spark.table(TARGET_TABLE)
total = df_bronze.count()
this_file_rows = df_bronze.filter(F.col("source") == FILE_NAME).count()
null_street_id = df_bronze.filter(F.col("street_id").isNull()).count()
null_date = df_bronze.filter(F.col("date").isNull()).count()
no_load_dt = df_bronze.filter(F.col("load_dt").isNull()).count()
no_source = df_bronze.filter(F.col("source").isNull()).count()

print(f"\n{'='*60}\n  INGESTION SUMMARY\n{'='*60}")
print(f"  Table                : {TARGET_TABLE}")
print(f"  Total rows           : {total:,}")
print(f"  Rows from {FILE_NAME}: {this_file_rows:,}")
print(f"{'='*60}\n  NULL / GRAIN CHECKS")
print(f"  null street_id       : {null_street_id:,}   (must be 0)")
print(f"  null date            : {null_date:,}   (must be 0)")
print(f"{'='*60}\n  AUDIT COLUMNS")
print(f"  missing load_dt      : {no_load_dt:,}   (must be 0)")
print(f"  missing source       : {no_source:,}   (must be 0)")
print(f"{'='*60}")
print(f"  Reader               : PySpark native XML (no Pandas)")
print(f"  Idempotency          : DELETE + append on source")
print(f"{'='*60}")

all_pass = (null_street_id == 0 and null_date == 0 and no_load_dt == 0
            and no_source == 0 and this_file_rows > 0)
print("  ALL CHECKS PASSED" if all_pass else "  WARNING: one or more checks failed — see above.")

display(spark.table(TARGET_TABLE).limit(10))